# CDC Seasonal Vaccine Prediction Model

The Goal of this Program is to try and predict the vaccination rate (as a %) of a given county using a machine learning model. Using the model, Counties with low vaccination rates can be identified ahead of time and the associated vaccination rate can be improved through advertising efforts.

This is done using data provided from the CDC. Follow the steps to generate and run the model.

## Instructions:
Visit the CDC Website and preview the dataset for "Influenza Vaccination Coverage for All Ages (6+ Months)" here:

https://data.cdc.gov/Flu-Vaccinations/Influenza-Vaccination-Coverage-for-All-Ages-6-Mont/vh55-3he6/about_data

Once there, Select "Export" on the right side of the page and download the dataset as a CSV file, leaving the file in your downloads folder

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

#User input filepath
file_path = input("Copy and Paste the file path here: ")

#Remove accidental quotes if the user pasted "C:\path\to\file.csv"
file_path = file_path.strip().strip('"').strip("'")

#Normalize or replace extra slashes if needed
file_path = file_path.replace("\\\\", "\\")

#Convert to a Path object and resolve issues like '~' home shortcuts
file_path = Path(file_path).expanduser().resolve()

#Check if the path actually exists and is a file
if file_path.is_file():
    #Print success if it is
    print(f"Success! Found file: {file_path.name}")
else:
    #print Error if it isn't
   print(f"Error: The path '{file_path}' does not point to a valid file.")

Copy and Paste the file path here:  "C:\Users\yousu\Downloads\Influenza_Vaccination_Coverage_for_All_Ages_(6+_Months)_20261020.csv"


Success! Found file: Influenza_Vaccination_Coverage_for_All_Ages_(6+_Months)_20261020.csv


Now run the rest of the cells to get find which counties in the US have the lowest vaccination rate

In [2]:
#Preprocessing
#Inport the CSV into Python as a dataframe
df = pd.read_csv(file_path)

#We are only interested in the Seasonal Vaccine so this filters the data to only the Seasonal Influenza Vaccine
df = df[df['Vaccine'] == 'Seasonal Influenza']

#We are only interested in counties so this statement filters out states/etc and leaves only counties
df = df[df['Geography Type'] == 'Counties']

#After filtering the above are 2 dimensions that are left in the Dataframe at this point for Dimension Type; "Age" and ">=18 Years". We want Age
df = df[df['Dimension Type'] == 'Age']

#There are counties that have the same name but are in different states. Concatenating County Name with FIPS yields a unique County Categorical field
df['County'] = df['Geography'].astype(str)+df['FIPS'].astype(str)

#Here there are a few features that have been exluded due to the fact that they are all the same value; Month(For counties, all months are '1'), Vaccine(we are testing only the Seasonal Influenza Vaccine), Dimension (The dimensions are all the same for counties (Age6+), and Sample Size(NA for Counties) exist but offer nothing to the model
df = df[['Season/Survey Year','County','Estimate (%)']]

#Need to cast "Season/Survey Year" as an integer to make it easier to filter later on
df['Season/Survey Year'] = df['Season/Survey Year'].astype(int)

#Convert Estimate to a number data type (it is object currently)
df['Estimate (%)'] = df['Estimate (%)'].astype(float)

# Convert County to a category type for our model later
df['County'] = df['County'].astype('category')

# Sort by year to ensure correct temporal ordering
df = df.sort_values(by='Season/Survey Year').reset_index(drop=True)

In [3]:
#Need to get the latest year +1 as our predict
last_year = int(df['Season/Survey Year'].max())
future_year = last_year + 1

In [4]:
# To capture true time-series dependency, we track what the percentage was last year and add it to our Dataframe as a new field
df['Estimate (%) Lag'] = df.groupby('County', observed = False)['Estimate (%)'].shift(1)

# Drop the first year since it won't have a lag value
df = df.dropna().reset_index(drop=True)

In [5]:
# We train on years up to the last, and test on the last year
train_set= df[df['Season/Survey Year'] < last_year]
test_set= df[df['Season/Survey Year'] >= last_year]

#Create Train set
y_train = train_set['Estimate (%)']
y_train = y_train.reset_index(drop=True)

X_train = train_set.drop(['Estimate (%)'], axis = 1)
X_train = X_train.reset_index(drop=True)

#Create Test Set
y_test = test_set['Estimate (%)']
y_test = y_test.reset_index(drop=True)

X_test = test_set.drop(['Estimate (%)'], axis = 1)
X_test = X_test.reset_index(drop=True)

In [6]:
import catboost as cb

#Initialize CatBoost Regressor
CBR_county_model = cb.CatBoostRegressor(
    iterations=1000, #number of iterations
    learning_rate=0.03, #Change per iteration
    depth=6, #max height of tree decision
    loss_function='RMSE', #the mathematical objective that guides how decision trees are built during training
    eval_metric='MAE', #a parameter used to measure how well a machine learning model is performing on a dataset
    random_seed=42,
    task_type='CPU', #specifies the hardware processing unit used to run the training and inference computations
    verbose=100,
)

In [7]:
#Fit the model passing cat_features explicitly
CBR_county_model.fit(
    X_train,
    y_train,
    cat_features=['County'],
    eval_set=(X_test, y_test),
    early_stopping_rounds=50,
    use_best_model=True,
)

0:	learn: 4.3901238	test: 4.9550242	best: 4.9550242 (0)	total: 166ms	remaining: 2m 45s
100:	learn: 2.0617747	test: 2.9280056	best: 2.9280056 (100)	total: 2.12s	remaining: 18.8s
200:	learn: 1.7867457	test: 2.8577393	best: 2.8577393 (200)	total: 4.32s	remaining: 17.2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 2.856785407
bestIteration = 221

Shrink model to first 222 iterations.


CatBoostRegressor(depth=6, eval_metric='MAE', iterations=1000, learning_rate=0.03, loss_function='RMSE', random_seed=42, task_type='CPU', verbose=100)

In [8]:
#Predict and clip percentages if bounded [0, 100]
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

predictions = CBR_county_model.predict(X_test)
predictions_clipped = np.clip(predictions, 0.0, 100.0)
rmse = np.sqrt(mean_squared_error(y_test, predictions_clipped))

print(f"Test RMSE for Year {last_year}: {rmse:.4f}")
print(f"Test MAE: {mean_absolute_error(y_test, predictions):.2f}%")
print(f"Test R2 Score: {r2_score(y_test, predictions):.2f}")

Test RMSE for Year 2022: 3.5896
Test MAE: 2.86%
Test R2 Score: 0.67


In [ ]:
#Find out which fields are the most important for the training process
importances = pd.Series(CBR_county_model.feature_importances_, index=X_train.columns)
print("\nFeature Importances:")
print(importances)

In [ ]:
#Future Data aka the year after the test year data
future_df = pd.DataFrame({
    'Season/Survey Year': [future_year]*len(test_set['County']),
    'County':test_set['County'],
    'Estimate (%) Lag': test_set.sort_values('County')['Estimate (%)'].values
})

In [ ]:
#Predicting the Future
#To predict The next year, we need the test percentages to act as our lag feature
future_preds = CBR_county_model.predict(future_df)

#Add it to our future_df
future_df['Predicted Estimate (%)'] = future_preds

In [ ]:
#Print the 3 counties with the lowest vaccination rate predicted by the model in the future year
low_vax = future_df.nsmallest(3, 'Predicted Estimate (%)')
low_vax['FIPS'] = low_vax['County'].str[-5:]
low_vax['County'] = low_vax['County'].str[:-5]
print('The 3 counties with lowest vaccination rate predicted in the year ' +str(future_year)+ ' are:')
print(low_vax[['County', 'FIPS', 'Predicted Estimate (%)']].to_string(index=False))